# Optimización de Hiperparámetros con Optuna
### …sin sobre-ajustar al pasado

`Fase 4 · Video 20` — serie **Forecasting de Ventas de la A a la Z**

Con la validación temporal del Video 19 como función objetivo, afinamos hiperparámetros de forma
honesta — y exponemos la trampa del **p-hacking**.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import optuna
from xgboost import XGBRegressor
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.backtest import temporal_splits
from src.metrics import wape
from src.events import get_event
from src.calendar_mx import quincena_factor_series, holiday_factor_series

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
y = df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]

dd = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
evm = np.array([(get_event(d.month, d.day).mult if get_event(d.month, d.day) else 1.0) for d in dd])
cal = pd.DataFrame({'ev': evm, 'qz': quincena_factor_series(dd), 'hol': holiday_factor_series(dd)[0]},
                   index=dd).resample('W-MON').mean().reindex(y.index)
F = pd.DataFrame({'lag52': y.shift(52), 'lag13': y.shift(13),
                  'woy': y.index.isocalendar().week.astype(int), 'month': y.index.month,
                  'ev': cal['ev'], 'qz': cal['qz'], 'hol': cal['hol']}, index=y.index)
data = F.assign(y=y.values).dropna()
X, t = data.drop(columns='y').values, data['y'].values

FINAL = 26                                   # test final, INTOCABLE
X_opt, t_opt = X[:-FINAL], t[:-FINAL]
X_test, t_test = X[-FINAL:], t[-FINAL:]
print(f'✅ Optimización: {len(t_opt)} semanas | test final intacto: {FINAL} semanas')

---
## 2. El montaje: Optuna + walk-forward

La pieza central es la **función objetivo**: para cada combinación de hiperparámetros, evaluamos el modelo
con **validación temporal** (`walk_forward` del Video 19) y devolvemos el WAPE medio. Optuna minimiza *ese*
número — así la búsqueda premia lo que generaliza al futuro, no lo que memoriza el pasado.

In [ ]:
folds = temporal_splits(len(t_opt), initial=90, horizon=13, step=13)

def cv_wape(params):
    scores = []
    for tr, te in folds:
        model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        model.fit(X_opt[tr], t_opt[tr])
        scores.append(wape(t_opt[te], np.clip(model.predict(X_opt[te]), 0, None)))
    return float(np.mean(scores))

def test_wape(params):
    model = XGBRegressor(**params, random_state=0, n_jobs=-1).fit(X_opt, t_opt)
    return wape(t_test, np.clip(model.predict(X_test), 0, None))

DEFAULT = dict(n_estimators=400, max_depth=6, learning_rate=0.05,
               subsample=0.8, colsample_bytree=0.8)
print(f'{len(folds)} folds temporales para la validación de cada trial.')
print(f'WAPE del XGBoost por defecto (en validación): {cv_wape(DEFAULT):.4f}')

---
## 3. La búsqueda: historia de optimización

Optuna propone configuraciones con un muestreador bayesiano (TPE): aprende de los trials anteriores para
buscar donde promete. Guardamos también el WAPE en el test final de cada trial **sin usarlo para decidir**
— nos servirá para exponer el p-hacking en la sección 6.

In [ ]:
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 600),
        max_depth=trial.suggest_int('max_depth', 2, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
    )
    trial.set_user_attr('test_wape', test_wape(params))   # solo para análisis, NO se optimiza
    return cv_wape(params)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=40, show_progress_bar=False)

cv_hist = [tr.value for tr in study.trials]
best_so_far = np.minimum.accumulate(cv_hist)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(range(1, len(cv_hist) + 1), cv_hist, 'o', color=BLUE, alpha=0.5, label='WAPE del trial (CV)')
ax.plot(range(1, len(cv_hist) + 1), best_so_far, color=RED, linewidth=2, label='mejor hasta ahora')
ax.axhline(cv_wape(DEFAULT), color='black', linestyle='--', linewidth=1, label='default')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('trial'); ax.set_ylabel('WAPE (validación temporal)')
ax.set_title('Historia de optimización de Optuna', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mejor WAPE (validación): {study.best_value:.4f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

---
## 4. Default vs. afinado (en validación)

In [ ]:
cv_default, cv_tuned = cv_wape(DEFAULT), study.best_value
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(['Default', 'Optuna'], [cv_default, cv_tuned], color=[BLUE, GREEN], width=0.5)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('WAPE en validación temporal', fontsize=13, fontweight='bold')
for i, v in enumerate([cv_default, cv_tuned]):
    ax.text(i, v, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Mejora en validación: {1 - cv_tuned / cv_default:.1%}')

---
## 5. La prueba honesta: el test final intacto

Optuna nunca vio estas últimas 26 semanas. Es el juez imparcial: si la mejora es real, debe **sostenerse**
aquí. Este paso —evaluar UNA sola vez en un test reservado— es lo que separa un resultado creíble de una
ilusión.

In [ ]:
res = pd.DataFrame({
    'Default':  [cv_wape(DEFAULT), test_wape(DEFAULT)],
    'Optuna':   [study.best_value, test_wape(study.best_params)],
}, index=['Validación (CV)', 'Test final (intacto)'])

fig, ax = plt.subplots(figsize=(9, 4.5))
res.T.plot(kind='bar', ax=ax, color=[ORANGE, GREEN])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('¿La mejora se sostiene fuera de la validación?', fontsize=13, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(res.round(4).to_string())
d_test, t_test_ = test_wape(DEFAULT), test_wape(study.best_params)
print(f'\nTest final — default {d_test:.1%} vs Optuna {t_test_:.1%} → mejora {1 - t_test_/d_test:+.1%}')
print('La mejora se sostiene en datos que Optuna jamás vio: la optimización fue honesta,')
print('porque su objetivo era la validación TEMPORAL, no un KFold aleatorio ni el test.')

---
## 6. La trampa del p-hacking

¿Y si en vez de elegir por validación hubiéramos elegido el trial con mejor **test**? Es hacer trampa
(mirar la respuesta), pero es un error facilísimo de cometer. Comparamos las dos formas de "elegir el
mejor".

In [ ]:
cv_scores = np.array([tr.value for tr in study.trials])
test_scores = np.array([tr.user_attrs['test_wape'] for tr in study.trials])

honest_idx = int(cv_scores.argmin())        # elegir por validación (correcto)
cheat_idx = int(test_scores.argmin())       # elegir por test (p-hacking)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(cv_scores, test_scores, color=BLUE, alpha=0.6, s=40)
ax.scatter(cv_scores[honest_idx], test_scores[honest_idx], color=GREEN, s=140,
           edgecolor='black', zorder=5, label='elegido por VALIDACIÓN (honesto)')
ax.scatter(cv_scores[cheat_idx], test_scores[cheat_idx], color=RED, s=140, marker='*',
           edgecolor='black', zorder=5, label='elegido por TEST (p-hacking)')
ax.set_xlabel('WAPE validación'); ax.set_ylabel('WAPE test final')
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Elegir por validación vs. por test (p-hacking)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Elegir por VALIDACIÓN → test = {test_scores[honest_idx]:.1%}  (lo que reportas honestamente)')
print(f'Elegir por TEST       → test = {test_scores[cheat_idx]:.1%}  (mirando la respuesta: mentira)')
print('\nEl número del p-hacking siempre será ≤ el honesto — porque estás optimizando sobre el')
print('propio test. En producción NO existe ese lujo: nadie te da las respuestas. Regla de oro:')
print('el test se toca UNA vez, al final, y no se usa jamás para elegir nada.')

---
## 7. Conclusiones y puente al Video 21

### Las reglas que te llevas

1. **La función objetivo de Optuna = tu validación temporal** (Video 19). Optimizar contra un KFold
   aleatorio o el test produce ganancias ficticias.
2. **Reserva un test final intacto** y tócalo **una sola vez**. Si la mejora se sostiene ahí, es real
   (aquí lo fue).
3. **Cuidado con el p-hacking:** con suficientes trials, alguno "gana" por azar. Elegir por test es
   engañarte a ti mismo.
4. **Pruning y early stopping** ahorran cómputo (Optuna descarta trials malos temprano) — úsalos con la
   validación correcta, no como atajo para peekar el test.
5. **Afinar da mejoras modestas pero reales** cuando el montaje es honesto — no esperes milagros, espera
   robustez.

### Cierre de la serie

Ya tenemos todo: datos, EDA, features, modelos, evaluación honesta y optimización. Solo falta **unirlo
todo** en un flujo que vaya del dato crudo a una decisión de negocio en pesos.

---

### Próximo video
**Video 21 — El framework completo: del dato crudo a la decisión de negocio**
Un dashboard que integra ingesta → features → modelo → reconciliación → métricas → recomendación de
inventario. El cierre de la serie.